# 8 puzzle Problem

Given a 3×3 board with 8 tiles (each numbered from 1 to 8) and one empty space, the objective is to place the numbers to match the final configuration using the empty space. We can slide four adjacent tiles (left, right, above, and below) into the empty space.

https://media.geeksforgeeks.org/wp-content/uploads/20240726131204/8-puzzle-Problem.webp

In [1]:
import heapq
from copy import deepcopy

class PuzzleState:
    def __init__(self, board, parent=None, move="", g=0, h=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.g = g  # Cost from start
        self.h = h  # Heuristic cost
        self.f = g + h  # Total cost
        self.empty_pos = self.find_empty()
    
    def find_empty(self):
        for i in range(3):
            for j in range(3):
                if self.board[i][j] == 0:
                    return (i, j)
        return None
    
    def __lt__(self, other):
        return self.f < other.f
    
    def __eq__(self, other):
        return self.board == other.board
    
    def __hash__(self):
        return hash(str(self.board))

def manhattan_distance(board, goal):
    distance = 0
    for i in range(3):
        for j in range(3):
            if board[i][j] != 0:
                value = board[i][j]
                for gi in range(3):
                    for gj in range(3):
                        if goal[gi][gj] == value:
                            distance += abs(i - gi) + abs(j - gj)
    return distance

def get_neighbors(state):
    neighbors = []
    i, j = state.empty_pos
    moves = [(-1, 0, "UP"), (1, 0, "DOWN"), (0, -1, "LEFT"), (0, 1, "RIGHT")]
    
    for di, dj, move_name in moves:
        ni, nj = i + di, j + dj
        if 0 <= ni < 3 and 0 <= nj < 3:
            new_board = deepcopy(state.board)
            new_board[i][j], new_board[ni][nj] = new_board[ni][nj], new_board[i][j]
            neighbors.append((new_board, move_name))
    
    return neighbors

def best_first_search(start, goal):
    start_state = PuzzleState(start, None, "", 0, manhattan_distance(start, goal))
    goal_state = PuzzleState(goal)
    
    open_list = []
    heapq.heappush(open_list, start_state)
    closed_set = set()
    
    while open_list:
        current = heapq.heappop(open_list)
        
        if current.board == goal:
            path = []
            while current.parent:
                path.append((current.move, current.board))
                current = current.parent
            path.reverse()
            return path
        
        closed_set.add(hash(str(current.board)))
        
        for neighbor_board, move in get_neighbors(current):
            if hash(str(neighbor_board)) not in closed_set:
                h = manhattan_distance(neighbor_board, goal)
                neighbor_state = PuzzleState(neighbor_board, current, move, current.g + 1, h)
                heapq.heappush(open_list, neighbor_state)
    
    return None

# Example usage
initial_state = [
    [1, 2, 3],
    [4, 0, 5],
    [6, 7, 8]
]

goal_state = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

print("Initial State:")
for row in initial_state:
    print(row)

print("\nGoal State:")
for row in goal_state:
    print(row)

print("\nSolving using Best First Search...")
solution = best_first_search(initial_state, goal_state)

if solution:
    print(f"\nSolution found in {len(solution)} moves:")
    for i, (move, board) in enumerate(solution, 1):
        print(f"\nStep {i}: {move}")
        for row in board:
            print(row)
else:
    print("\nNo solution found!")

Initial State:
[1, 2, 3]
[4, 0, 5]
[6, 7, 8]

Goal State:
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]

Solving using Best First Search...

Solution found in 14 moves:

Step 1: RIGHT
[1, 2, 3]
[4, 5, 0]
[6, 7, 8]

Step 2: DOWN
[1, 2, 3]
[4, 5, 8]
[6, 7, 0]

Step 3: LEFT
[1, 2, 3]
[4, 5, 8]
[6, 0, 7]

Step 4: LEFT
[1, 2, 3]
[4, 5, 8]
[0, 6, 7]

Step 5: UP
[1, 2, 3]
[0, 5, 8]
[4, 6, 7]

Step 6: RIGHT
[1, 2, 3]
[5, 0, 8]
[4, 6, 7]

Step 7: DOWN
[1, 2, 3]
[5, 6, 8]
[4, 0, 7]

Step 8: RIGHT
[1, 2, 3]
[5, 6, 8]
[4, 7, 0]

Step 9: UP
[1, 2, 3]
[5, 6, 0]
[4, 7, 8]

Step 10: LEFT
[1, 2, 3]
[5, 0, 6]
[4, 7, 8]

Step 11: LEFT
[1, 2, 3]
[0, 5, 6]
[4, 7, 8]

Step 12: DOWN
[1, 2, 3]
[4, 5, 6]
[0, 7, 8]

Step 13: RIGHT
[1, 2, 3]
[4, 5, 6]
[7, 0, 8]

Step 14: RIGHT
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]


In [2]:
def a_star_search(start, goal):
    start_state = PuzzleState(start, None, "", 0, manhattan_distance(start, goal))
    goal_state = PuzzleState(goal)
    
    open_list = []
    heapq.heappush(open_list, start_state)
    closed_set = set()
    
    while open_list:
        current = heapq.heappop(open_list)
        
        if current.board == goal:
            path = []
            while current.parent:
                path.append((current.move, current.board))
                current = current.parent
            path.reverse()
            return path
        
        closed_set.add(hash(str(current.board)))
        
        for neighbor_board, move in get_neighbors(current):
            if hash(str(neighbor_board)) not in closed_set:
                g = current.g + 1  # Actual cost from start
                h = manhattan_distance(neighbor_board, goal)  # Heuristic cost
                neighbor_state = PuzzleState(neighbor_board, current, move, g, h)
                heapq.heappush(open_list, neighbor_state)
    
    return None

print("\nSolving using A* Search...")
solution_astar = a_star_search(initial_state, goal_state)

if solution_astar:
    print(f"\nSolution found in {len(solution_astar)} moves:")
    for i, (move, board) in enumerate(solution_astar, 1):
        print(f"\nStep {i}: {move}")
        for row in board:
            print(row)
else:
    print("\nNo solution found!")


Solving using A* Search...

Solution found in 14 moves:

Step 1: RIGHT
[1, 2, 3]
[4, 5, 0]
[6, 7, 8]

Step 2: DOWN
[1, 2, 3]
[4, 5, 8]
[6, 7, 0]

Step 3: LEFT
[1, 2, 3]
[4, 5, 8]
[6, 0, 7]

Step 4: LEFT
[1, 2, 3]
[4, 5, 8]
[0, 6, 7]

Step 5: UP
[1, 2, 3]
[0, 5, 8]
[4, 6, 7]

Step 6: RIGHT
[1, 2, 3]
[5, 0, 8]
[4, 6, 7]

Step 7: DOWN
[1, 2, 3]
[5, 6, 8]
[4, 0, 7]

Step 8: RIGHT
[1, 2, 3]
[5, 6, 8]
[4, 7, 0]

Step 9: UP
[1, 2, 3]
[5, 6, 0]
[4, 7, 8]

Step 10: LEFT
[1, 2, 3]
[5, 0, 6]
[4, 7, 8]

Step 11: LEFT
[1, 2, 3]
[0, 5, 6]
[4, 7, 8]

Step 12: DOWN
[1, 2, 3]
[4, 5, 6]
[0, 7, 8]

Step 13: RIGHT
[1, 2, 3]
[4, 5, 6]
[7, 0, 8]

Step 14: RIGHT
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]
